In [3]:
import requests

from termcolor import colored

import pprint

from IPython.display import Image, display

def get_game_title(pokemon_number):
    pokemon_forms = requests.get(f"https://pokeapi.co/api/v2/pokemon-form/{pokemon_number}/").json()

    pokemon_version = pokemon_forms['version_group']['name']

    if pokemon_version is None:
        return 'Game title not found'

    list_of_words = pokemon_version.split('-')

    num_of_words = len(list_of_words)

    if num_of_words == 2:
        return(list_of_words[0].title() + " and " + list_of_words[1].title())
    
    elif num_of_words == 4:
        return(list_of_words[0].title() + ' ' + list_of_words[1].title() + " and " + list_of_words[2].title() + ' ' + list_of_words[3].title())

    return 'Game title not found'


pokemon_number = 1

response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{pokemon_number}")

pokemon_full_data = response.json()

sprite = pokemon_full_data['sprites']['front_default']
display(Image(url=sprite,width = 200))


pokemon_name = pokemon_full_data['name']
print(colored(pokemon_name.title(),attrs=["bold"]))

#Types
pokemon_type_1 = pokemon_full_data['types'][0]['type']['name']

try:
    pokemon_type_2 = pokemon_full_data['types'][1]['type']['name']
except IndexError: 
    pokemon_type_2 = False

if pokemon_type_2:
    print(pokemon_type_1.title(), " / ", pokemon_type_2.title())

else:
    print(pokemon_type_1.title())


# Game title
print()
print(colored("Game introduced:",attrs=["bold"]))
print(get_game_title(pokemon_number))


# Stats
pokemon_stats = pokemon_full_data['stats']
pokemon_hp = pokemon_full_data['stats'][0]['base_stat']
pokemon_attack = pokemon_full_data['stats'][1]['base_stat']
pokemon_defence = pokemon_full_data['stats'][2]['base_stat']
pokemon_sp_attack = pokemon_full_data['stats'][3]['base_stat']
pokemon_sp_defence = pokemon_full_data['stats'][4]['base_stat']
pokemon_speed = pokemon_full_data['stats'][5]['base_stat']

print()
print(colored("Stats: ",attrs=["bold"]))
print("HP:", pokemon_hp)
print("Attack:", pokemon_attack)
print("Defence:", pokemon_defence)
print("Special attack:", pokemon_sp_attack)
print("Speed:", pokemon_speed)
print()

# Defending vs a specific type
print(colored("Defending against a specific type:",attrs=["bold"]))


# Pulling all types from PokeAPI
type_API = requests.get(f"https://pokeapi.co/api/v2/type/").json()
all_types = type_API['results'] # all_types is a LIST
type_names = [type_info['name'] for type_info in all_types]

# Each type's double_damage_from/to and no_damage_to
#Test with one type - then expand to loop against all types to just pull out double/half/no effects
type_name = "electric"
print("Selected move type:",type_name.title())
type_api = requests.get(f"https://pokeapi.co/api/v2/type/{type_name}").json()
relations = type_api['damage_relations']
damage_profile = {
    "double_damage_from": [entry['name'] for entry in relations['double_damage_from']],
    "double_damage_to": [entry['name'] for entry in relations['double_damage_to']],
    "half_damage_from": [entry['name'] for entry in relations['half_damage_from']],
    "half_damage_to": [entry['name'] for entry in relations['half_damage_to']],
    "no_damage_from": [entry['name'] for entry in relations['no_damage_from']],
    "no_damage_to": [entry['name'] for entry in relations['no_damage_to']],
}




# Calculating move effectiveness v Pokemon
    # Move v Pokemon's type 1

if pokemon_type_1 in damage_profile['double_damage_to']:
    effectiveness = 2
elif pokemon_type_1 in damage_profile['half_damage_to']:
    effectiveness = 0.5
elif pokemon_type_1 in damage_profile['no_damage_to']:
    effectiveness = 0
else:
    effectiveness = 1

print("Effectiveness of",type_name,"move vs",pokemon_type_1,"type:",effectiveness)

    # Move v Pokemon's type 2

if pokemon_type_2 == False:
    effectiveness_2 = "No type 2"
elif pokemon_type_2 in damage_profile['double_damage_to']:
    effectiveness_2 = 2
elif pokemon_type_2 in damage_profile['half_damage_to']:
    effectiveness_2 = 0.5
elif pokemon_type_2 in damage_profile['no_damage_to']:
    effectiveness_2 = 0
else:
    effectiveness_2 = 1

# Calculation for overall effectiveness

if pokemon_type_2:
    matchup = effectiveness * effectiveness_2
else:
    matchup = effectiveness

if pokemon_type_2:
    print("Effectiveness of", type_name, "move vs", pokemon_type_2, "type:", effectiveness_2)

if pokemon_type_2 != False:
    print(colored("Overall:",attrs=["bold"]),type_name.title(),"type move effectiveness vs dual type:",matchup)




# Get type strengths and weaknesses 
# note: the following isn't correct. Need to fix logic for dual types (currently just pulls stengths/weaknesses from each type, rather than applying the multiplication formula)

type_1_url = pokemon_full_data['types'][0]['type']['url']
type_1_response = requests.get(pokemon_full_data['types'][0]['type']['url'])
type_1_data = type_1_response.json()

# weak_vs = type_1_data['damage_relations']['double_damage_from']
# print()
# print(colored("Weak vs: ",attrs=["bold"]))
# weak_list = [element['name'] for element in weak_vs]
# sorted_weak = sorted(weak_list)
# print(", ".join(sorted_weak).title())
# print()


# strong_vs = type_1_data['damage_relations']['double_damage_to']
# print(colored("Strong vs: ",attrs=["bold"]))
# strong_list = [element['name'] for element in strong_vs]
# sorted_strong = sorted(strong_list)
# print(", ".join(sorted_strong).title())


Bulbasaur
Grass  /  Poison

Game introduced:
Red and Blue

Stats: 
HP: 45
Attack: 49
Defence: 49
Special attack: 65
Speed: 45

Defending against a specific type:
Selected move type: Electric
Effectiveness of electric move vs grass type: 0.5
Effectiveness of electric move vs poison type: 1
Overall: Electric type move effectiveness vs dual type: 0.5
